# Data Modelling

- Among the objects JPL has placed on the Sentry risk list, which ones get cleared — and does the orbit itself predict it, independent of how much telescope time the object received?

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score

ROOT = os.path.abspath("..")
sys.path.append(os.path.join(ROOT, "src"))

from ml.dataset import load_data



#### 1. Load the data
- We use here the SQL query from 'dataset.sql' and then load it using the python function from 'dataset.py'.

In [2]:
df = load_data()
df.shape

(5032, 29)

In [3]:
df.head()

,spkid,pdes,removed,moid,q,a,e,i,om,w,...,days_since_last_obs,rms,condition_code,sigma_a,sigma_e,sigma_q,sigma_i,sigma_w,sigma_tp,class
0,1003080,2009 WX51,1,0.009330,0.800,3.0800,0.7403,9.59,31.76,118.02,...,6041,0.49193,4,1.000000e-04,9.400000e-06,2.000000e-06,1.700000e-04,3.700000e-04,1.100000e-04,JFc
1,1004036,2024 OC2,1,0.020300,0.593,2.6260,0.7742,7.67,136.62,258.54,...,667,0.61230,4,2.800000e-05,2.800000e-06,1.000000e-06,9.600000e-05,3.400000e-04,6.000000e-05,JFc
2,20029075,29075,0,0.039200,0.836,1.6980,0.5076,12.16,356.54,224.82,...,11,0.48551,0,1.000000e-09,1.200000e-08,2.000000e-08,2.600000e-07,5.500000e-06,4.600000e-06,APO
3,20099942,99942,1,0.000108,0.746,0.9224,0.1911,3.34,203.89,126.68,...,1666,0.25657,0,1.600000e-09,1.800000e-09,2.900000e-09,9.900000e-08,3.300000e-06,7.100000e-07,ATE
4,20101955,101955,0,0.003220,0.897,1.1260,0.2037,6.03,2.06,66.22,...,2219,0.29146,0,2.000000e-10,7.000000e-11,2.300000e-10,6.900000e-09,6.400000e-08,2.300000e-08,APO


#### 2. Count NaN
- We need to next count the null values in each column so that we can impute/drop if necessary.

In [4]:
df.isna().sum()

spkid                     0
pdes                      0
removed                   0
moid                      0
q                         0
a                         0
e                         0
i                         0
om                        0
w                         0
h                         3
h_sigma                2255
has_h_sigma               0
has_diameter              0
n_obs_used                0
n_del_obs_used            0
n_dop_obs_used            0
has_radar                 0
data_arc                  0
days_since_last_obs       0
rms                       0
condition_code            0
sigma_a                   0
sigma_e                   0
sigma_q                   0
sigma_i                   0
sigma_w                   0
sigma_tp                  0
class                     0
dtype: int64

- There are just two columns 'h' and 'h_sigma' which show null/zero values. We will need to impute these based on their 'median' values.

#### 3. Train-Test split

- The features that go into model building will not naturally carry their ID, which in this case, are: "spkid", "pdes", "removed".
- Also, we are ignoring comets.

In [5]:
asteroid_classes = ['APO', 'AMO', 'ATE', 'IEO']
df = df[df['class'].isin(asteroid_classes)].copy()

features = df.drop(columns=["spkid", "pdes", "removed"])
target = df["removed"]

print(features.shape)

(5030, 26)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, stratify=target, random_state=42)
print(X_train.shape, X_test.shape)

(4024, 26) (1006, 26)


#### 4. Pre-processing

- Columns 'has_h_sigma', 'has_diameter', 'has_radar' show boolean nature, however, these need to be transformed into integer type.

In [7]:
bools = ['has_h_sigma', 'has_diameter', 'has_radar']
features[bools] = features[bools].astype(int)

- Imputation on 'h' and 'h_signma'

In [8]:
impute = SimpleImputer(missing_values=np.nan, strategy='median')

X_train[['h', 'h_sigma']] = impute.fit_transform(X_train[['h', 'h_sigma']])
X_test[['h', 'h_sigma']] = impute.transform(X_test[['h', 'h_sigma']])

- Standardization on the rest of the columns

In [9]:
cols = ['moid', 'q', 'a', 'e', 'i', 'om', 'w', 'h', 'h_sigma',
        'n_obs_used', 'n_del_obs_used', 'n_dop_obs_used',
        'data_arc', 'days_since_last_obs', 'rms', 'condition_code',
        'sigma_a', 'sigma_e', 'sigma_q', 'sigma_i', 'sigma_w', 'sigma_tp']

scaler = StandardScaler()

X_train[cols] = scaler.fit_transform(X_train[cols])
X_test[cols] = scaler.transform(X_test[cols])

- Perform One Hot Encoding on the column 'class' which has categorial values.
- But we want to focus only on asteroids and not the comets.

In [10]:

ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

class_train = pd.DataFrame(
    ohe.fit_transform(X_train[['class']]),
    columns=ohe.get_feature_names_out(['class']),
    index=X_train.index
)

class_test = pd.DataFrame(
    ohe.transform(X_test[['class']]),
    columns=ohe.get_feature_names_out(['class']),
    index=X_test.index
)

X_train = pd.concat([X_train.drop(columns=['class']), class_train], axis=1)
X_test = pd.concat([X_test.drop(columns=['class']), class_test], axis=1)
print(X_train.shape)

(4024, 28)


#### 5. Training

- We use Logistic Regression in order to predict if an object will be removed from the sentry list or not.

In [11]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

print("Training accuracy: ", lr.score(X_train, y_train))
print("Testing accuracy: ", lr.score(X_test, y_test))

print("Training PR-AUC: ", average_precision_score(y_train, lr.predict_proba(X_train)[:, 1]))
print("Testing PR-AUC: ", average_precision_score(y_test, lr.predict_proba(X_test)[:, 1]))

Training accuracy:  0.8156063618290258


Testing accuracy:  0.8011928429423459
Training PR-AUC:  0.9230602507975955
Testing PR-AUC:  0.9106614764154158


In [12]:
ORBITAL = ['moid','q','a','e','i','om','w','h']
OBSERVATIONAL = ['h_sigma','has_h_sigma','has_diameter','n_obs_used',
                 'n_del_obs_used','n_dop_obs_used','has_radar','data_arc',
                 'days_since_last_obs','rms','condition_code',
                 'sigma_a','sigma_e','sigma_q','sigma_i','sigma_w','sigma_tp']

for name, cols in [("orbital only", ORBITAL), ("observational only", OBSERVATIONAL),("both", ORBITAL + OBSERVATIONAL)]:
    model = LogisticRegression()
    model.fit(X_train[cols], y_train)

    print("%-20s test PR-AUC: %.4f" % (name, average_precision_score(y_test, model.predict_proba(X_test[cols])[:, 1])))

orbital only         test PR-AUC: 0.8805


observational only   test PR-AUC: 0.8110
both                 test PR-AUC: 0.9109


In [13]:
y_pred = lr.predict(X_test)

display(pd.crosstab(y_test,y_pred, rownames=['True'], colnames=['Prediction']))
print(classification_report(y_test, y_pred))

Prediction,0,1
True,,
0,374,71
1,129,432


              precision    recall  f1-score   support

           0       0.74      0.84      0.79       445
           1       0.86      0.77      0.81       561

    accuracy                           0.80      1006
   macro avg       0.80      0.81      0.80      1006
weighted avg       0.81      0.80      0.80      1006

